# Results processing

### 1. Load

In [ ]:
MODEL_SHORT = "QwQ-32B"
DATASET_COL = "qwq"
MODEL_FULL_NAME = "Qwen/QwQ-32B"
MODEL_PARAMS = {"temperature": 1.0, "top_p": 1.0}
IS_THINKING_MODEL = True

RESPONSES_PATH = f"./data/responses/{MODEL_SHORT}/responses.tsv"
TRANSCRIPTS_PATH = "./data/ted_talks_transcripts.tsv"
OUTPUT_DATASET_DIR = "./data/dataset"

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

In [2]:
import json
import re
import pandas as pd

pd.set_option("display.max_colwidth", None)

responses = pd.read_csv(RESPONSES_PATH, sep="\t")
transcripts = pd.read_csv(TRANSCRIPTS_PATH, sep="\t")

print(f"Model: {MODEL_SHORT}  |  odpowiedzi: {len(responses)}")
print(f"Per jezyk: {responses['target_lang'].value_counts().to_dict()}")
responses.head()

Model: QwQ-32B  |  odpowiedzi: 6
Per jezyk: {'zh': 1, 'fi': 1, 'fr': 1, 'he': 1, 'ja': 1, 'pl': 1}


id  \
0  zh--adam_grant_how_to_stop_languishing_and_start_finding_flow   
1  fi--adam_grant_how_to_stop_languishing_and_start_finding_flow   
2  fr--adam_grant_how_to_stop_languishing_and_start_finding_flow   
3  he--adam_grant_how_to_stop_languishing_and_start_finding_flow   
4  ja--adam_grant_how_to_stop_languishing_and_start_finding_flow   

                                                        slug target_lang  \
0  adam_grant_how_to_stop_languishing_and_start_finding_flow          zh   
1  adam_grant_how_to_stop_languishing_and_start_finding_flow          fi   
2  adam_grant_how_to_stop_languishing_and_start_finding_flow          fr   
3  adam_grant_how_to_stop_languishing_and_start_finding_flow          he   
4  adam_grant_how_to_stop_languishing_and_start_finding_flow          ja   

  target_lang_name  \
0          Chinese   
1          Finnish   
2           French   
3           Hebrew   
4         Japanese   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
PREVIEW_LANG = "pl"

_preview = responses[responses["target_lang"] == PREVIEW_LANG]
if len(_preview):
    print(f"{PREVIEW_LANG}  |  {_preview.iloc[0]['id']}")
    print("=" * 80)
    print(_preview.iloc[0]["response"])
else:
    print(f"Brak odpowiedzi dla jezyka '{PREVIEW_LANG}'")

### Narzedzia SRT

In [ ]:
TS_RE = re.compile(r"\d{2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*\d{2}:\d{2}:\d{2}[,.]\d{1,3}")


def strip_think(text):
    text = str(text)
    if "</think>" in text:
        text = text.split("</think>")[-1]
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)
    return text.strip()


def extract_srt(text):
    text = str(text)
    text = re.sub(r"```[a-zA-Z]*\n?", "", text)

    lines = text.split("\n")
    ts_idxs = [i for i, l in enumerate(lines) if TS_RE.search(l)]
    if not ts_idxs:
        return ""

    first_ts = ts_idxs[0]
    start = first_ts - 1 if first_ts > 0 and lines[first_ts - 1].strip().isdigit() else first_ts
    return "\n".join(lines[start:]).strip()


def normalize_srt(text):
    blocks = []
    for raw in re.split(r"\n\s*\n", str(text).strip()):
        block_lines = [l for l in raw.split("\n") if l.strip()]
        ts_pos = next((i for i, l in enumerate(block_lines) if TS_RE.search(l)), None)
        if ts_pos is None:
            continue
        ts_line = block_lines[ts_pos].strip()
        body = [l.strip() for l in block_lines[ts_pos + 1:] if l.strip()]
        if not body:
            continue
        num = block_lines[ts_pos - 1].strip() if ts_pos > 0 and block_lines[ts_pos - 1].strip().isdigit() else str(len(blocks) + 1)
        blocks.append(f"{num}\n{ts_line}\n" + "\n".join(body))
    return "\n\n".join(blocks)


def clean_srt(text):
    return normalize_srt(extract_srt(text))

### 2. Czyszczenie - modele myslace

In [ ]:
assert IS_THINKING_MODEL, "Model nie jest oznaczony jako myslacy - uzyj sekcji 3."

responses["cleaned"] = responses["response"].apply(lambda t: clean_srt(strip_think(t)))

empty = (responses["cleaned"].str.len() == 0).sum()
print(f"Wyczyszczono {len(responses)} odpowiedzi  |  pustych po czyszczeniu: {empty}")
print("=" * 80)
print(responses[responses["target_lang"] == PREVIEW_LANG].iloc[0]["cleaned"][:1500])

### 3. Czyszczenie - modele zwykle

In [ ]:
assert not IS_THINKING_MODEL, "Model jest myslacy - uzyj sekcji 2."

responses["cleaned"] = responses["response"].apply(clean_srt)

empty = (responses["cleaned"].str.len() == 0).sum()
print(f"Wyczyszczono {len(responses)} odpowiedzi  |  pustych po czyszczeniu: {empty}")
print("=" * 80)
print(responses[responses["target_lang"] == PREVIEW_LANG].iloc[0]["cleaned"][:1500])

### 4. Podzial na jezyki

In [ ]:
import os

assert "cleaned" in responses.columns, "Brak kolumny 'cleaned' - uruchom sekcje 2 albo 3."

SPLIT_DIR = f"./data/responses/{MODEL_SHORT}"
os.makedirs(SPLIT_DIR, exist_ok=True)

lang_frames = {}
for lang in TARGET_LANGUAGES:
    df_lang = responses[responses["target_lang"] == lang].copy()
    if df_lang.empty:
        continue
    df_lang["slug"] = df_lang["id"].str.replace(f"^{lang}--", "", regex=True)
    lang_frames[lang] = df_lang
    out_path = f"{SPLIT_DIR}/{MODEL_SHORT}__{lang}.tsv"
    df_lang.to_csv(out_path, sep="\t", index=False)
    print(f"  {lang}: {len(df_lang)} wierszy -> {out_path}")

### 5. Format do datasetu

In [ ]:
assert "lang_frames" in dir(), "Brak 'lang_frames' - uruchom sekcje 4."

AVAILABLE_LANGS = "en," + ",".join(TARGET_LANGUAGES)

FIXED_COLS = ["id", "available_lang_vers", "title", "article_revision_id",
              "summary", "en_source", "prompt"]

meta = transcripts.set_index("slug")


def build_schema(lang, model_cols):
    cols = [
        {"name": "id", "description": "Talk slug (ten sam dla danego talku we wszystkich jezykach)",
         "language_model": None, "cleaned": None},
        {"name": "available_lang_vers", "description": "Kody wszystkich dostepnych wersji jezykowych talku",
         "language_model": None, "cleaned": None},
        {"name": "title", "description": "Tytul TED talku", "language_model": None, "cleaned": None},
        {"name": "article_revision_id", "description": "Brak odpowiednika dla TED (puste)",
         "language_model": None, "cleaned": None},
        {"name": "summary", "description": f"Oficjalne tlumaczenie TED ({lang}) w formacie SRT (referencja)",
         "language_model": None, "cleaned": None},
        {"name": "en_source", "description": "Zrodlowa transkrypcja EN w formacie SRT (wejscie do tlumaczenia)",
         "language_model": None, "cleaned": None},
        {"name": "prompt", "description": "Prompt uzyty do wygenerowania odpowiedzi modelu",
         "language_model": None, "cleaned": None},
    ]
    for name, cleaned in model_cols:
        cols.append({
            "name": name,
            "description": f"{MODEL_SHORT} response" + (" (cleaned version)" if cleaned else ""),
            "language_model": {"name": MODEL_FULL_NAME, "parameters": MODEL_PARAMS},
            "cleaned": cleaned,
        })
    return {"data_file": {"lang": lang, "name": {"stem": f"{lang}_ted_data", "suffix": ".csv"}},
            "columns": cols}


for lang, df_lang in lang_frames.items():
    out_dir = f"{OUTPUT_DATASET_DIR}/{lang}"
    os.makedirs(out_dir, exist_ok=True)

    rows = []
    for _, r in df_lang.iterrows():
        slug = r["slug"]
        m = meta.loc[slug] if slug in meta.index else None
        rows.append({
            "id": slug,
            "available_lang_vers": AVAILABLE_LANGS,
            "title": (m["Ted talk title"] if m is not None else ""),
            "article_revision_id": "",
            "summary": (m[f"{lang}_timed"] if m is not None else ""),
            "en_source": (m["en_timed"] if m is not None else ""),
            "prompt": r["prompt"],
            DATASET_COL: r["response"],
            f"{DATASET_COL}__cleaned": r["cleaned"],
        })

    out_df = pd.DataFrame(rows, columns=FIXED_COLS + [DATASET_COL, f"{DATASET_COL}__cleaned"])
    csv_path = f"{out_dir}/{lang}_ted_data.csv"
    schema_path = f"{out_dir}/{lang}_ted_schema.json"
    out_df.to_csv(csv_path, index=False)

    model_cols = [(DATASET_COL, False), (f"{DATASET_COL}__cleaned", True)]
    with open(schema_path, "w", encoding="utf-8") as f:
        json.dump(build_schema(lang, model_cols), f, ensure_ascii=False, indent=2)

    print(f"  {lang}: {len(out_df)} wierszy -> {csv_path}  (+ schema)")